# Análisis exploratorio de datos

# 1. Importación de librerías

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests
from shapely.geometry import Polygon
from io import StringIO


# 2. Obtención y carga de datos

Los datos se descargan desde la API pública de USGS, que permite consultar su catálogo de sismos filtrando por fecha y por ubicación. Los parámetros de la consulta son:

- **Período:** del 15/09/2016 al 15/09/2026 (diez años).
- **Área:** un rectángulo entre las latitudes 90° S y 21,7° S y las longitudes 75° O y 25° O. Cubre todo el territorio argentino, incluido el sector antártico. Como es un rectángulo, también incluye zonas de países vecinos y del océano, que se descartan en la sección 3.
- **Límite:** hasta 20.000 eventos, suficiente para no perder registros del período.

La respuesta llega en formato CSV y se carga en un DataFrame.

In [ ]:
# Importo los datos

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "csv",
    "starttime": "2016-09-15",
    "endtime": "2026-09-15",
    "minlatitude": -90,
    "maxlatitude": -21.7,
    "minlongitude": -75,
    "maxlongitude": -25,
    "limit": 20000
}

response = requests.get(url, params=params)
response.raise_for_status()



In [ ]:
# DataFrame

df = pd.read_csv(StringIO(response.text))

##Análisis superficial de los datos para verificar su integridad.

In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])


In [ ]:
pd.set_option('display.max_columns', None)

df.head(3)

Cada fila del conjunto es un sismo. El significado de cada columna, según la [documentación de USGS](https://earthquake.usgs.gov/data/comcat/data-eventterms.php), y qué se decide hacer con cada una después del análisis:

| Variable | Descripción | Unidad | Decisión |
|---|---|---|---|
| `time` | Fecha y hora en que ocurrió el evento (UTC). | — | Se convierte a tipo fecha |
| `latitude` | Latitud del epicentro. Valores negativos indican hemisferio sur. | grados | Se conserva |
| `longitude` | Longitud del epicentro. Valores negativos indican oeste de Greenwich. | grados | Se conserva |
| `depth` | Profundidad del hipocentro, es decir, del punto donde se origina la ruptura. | km | Se conserva |
| `mag` | Magnitud del evento: medida de la energía liberada. | — | Se conserva |
| `magType` | Método o escala con que se calculó la magnitud (`mb` = ondas de cuerpo, `mww` = magnitud de momento fase W, `ml` = magnitud local, etc.). | — | Se conserva |
| `nst` | Cantidad de estaciones sísmicas usadas para localizar el evento. | estaciones | **Se descarta:** 60 % de faltantes |
| `gap` | Brecha azimutal: el mayor ángulo entre dos estaciones vecinas vistas desde el epicentro. Cuanto menor, más confiable la ubicación (por encima de 180° es poco confiable). | grados | Se imputan los faltantes |
| `dmin` | Distancia horizontal entre el epicentro y la estación más cercana (1° ≈ 111,2 km). | grados | Se imputan los faltantes |
| `rms` | Error de ajuste entre los tiempos de llegada observados y los calculados. Cuanto menor, mejor la localización. | segundos | Se imputan los faltantes |
| `net` | Red sísmica que aportó la solución preferida del evento. | — | **Se descarta:** el 99,97 % es `us` |
| `id` | Identificador único del evento. | — | Se conserva |
| `updated` | Fecha y hora de la última actualización del registro. | — | Se convierte a tipo fecha |
| `place` | Descripción textual de la ubicación. | — | Se conserva |
| `type` | Tipo de evento (sismo, explosión de cantera, etc.). | — | **Se descarta:** el 100 % es `earthquake` |
| `horizontalError` | Incertidumbre de la ubicación horizontal. | km | Se imputan los faltantes |
| `depthError` | Incertidumbre de la profundidad. | km | Se conserva (sin faltantes) |
| `magError` | Incertidumbre de la magnitud. | — | Se imputan los faltantes |
| `magNst` | Cantidad de estaciones usadas para calcular la magnitud. | estaciones | Se imputan los faltantes |
| `status` | Estado de revisión: `automatic` o `reviewed` (revisado por un sismólogo). | — | **Se descarta:** el 100 % es `reviewed` |
| `locationSource` | Red que calculó la ubicación. | — | **Se descarta:** el 99,9 % es `us` |
| `magSource` | Red que calculó la magnitud. | — | **Se descarta:** el 96,5 % es `us` |

Las variables `nst`, `gap`, `dmin`, `rms`, `horizontalError`, `depthError`, `magError` y `magNst` son **variables de calidad**: no describen el sismo en sí, sino qué tan confiable es su medición.

La columna "Decisión" resume las conclusiones de las secciones 4.2 y 4.4; ahí está el análisis que las justifica.

In [ ]:
# Guardado de datos originales en .csv

df.to_csv("../data/raw/sismos_usgs.csv", index=False)


##Visualización de los datos obtenidos

**Fuente:** https://www.ign.gob.ar/NuestrasActividades/InformacionGeoespacial/CapasSIG


In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326"
)

In [ ]:
# lectura archivo SHP para graficar: Sudamerica(Poligono) y Argentina(Linea) (IGN)

ign_sudam = gpd.read_file("../data/raw/geodata/referencias.shp")
ign_arg   = gpd.read_file("../data/raw/geodata/limites.shp")


In [ ]:

fig, ax = plt.subplots(figsize=(10, 12))

ign_sudam.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.4
)

ign_arg.plot(
    ax=ax,
    color="black",
    linewidth=0.5
)

gdf.plot(
    ax=ax,
    column="mag",
    cmap="viridis",
    markersize=5,
    legend=True,
    legend_kwds={
        "orientation": "horizontal",
        "shrink": 0.4,
        "pad": 0.04
    }
)

# Zoom sobre Argentina y alrededores
ax.set_xlim(-76, -24)
ax.set_ylim(-90, -20)

ax.set_title(
    "Sismos registrados por el USGS en el área de estudio",
    fontsize=14,
    pad=12
)

plt.show()

**Figura 1. Eventos sísmicos registrados por el USGS dentro del área rectangular inicial de consulta.**

 El mapa incluye zonas externas a Argentina, ya que la API se consultó utilizando un rectángulo geográfico amplio. Estos eventos serán filtrados posteriormente mediante un polígono más preciso del territorio argentino.

# 3. Definición del área de estudio

Para quedarse solo con los sismos de Argentina se arma un polígono que une tres zonas:

- las provincias argentinas (capa del IGN),
- la plataforma continental argentina (capa del IGN), para incluir los sismos en el mar,
- el sector antártico argentino, entre los meridianos 25° O y 74° O, al sur del paralelo 60° S.

Se conservan los eventos cuyo epicentro cae dentro de ese polígono. El resultado se guarda en `df_arg`, que es el conjunto que se analiza en el resto del notebook.

In [ ]:
# lectura de archivos SHP: provincias y plataforma continental (IGN)

ign_provincias = gpd.read_file("../data/raw/geodata/provinciaPolygon.shp")
ign_plataforma = gpd.read_file("../data/raw/geodata/plataforma_continentalPolygon.shp")


# armado del sector antártico argentino

coordenadas_antartida = [
    (-74.0, -60.0),  # Esquina Noroeste
    (-25.0, -60.0),  # Esquina Noreste
    (-25.0, -90.0),  # Polo Sur (Límite Este)
    (-74.0, -90.0),  # Polo Sur (Límite Oeste)
    (-74.0, -60.0)   # Cierre
]

poligono_antartico = Polygon(coordenadas_antartida)

# Unión de las 3 geometrías en un solo polígono

poligono_nacional = (
    ign_provincias.union_all()
    .union(ign_plataforma.union_all())
    .union(poligono_antartico)
)



In [ ]:
# filtrado de puntos ubicados dentro del polígono

gdf_arg = gdf[gdf.geometry.within(poligono_nacional)]

df_arg = pd.DataFrame(gdf_arg.drop(columns='geometry'))


Visualización de los sismos en Argentina entre 2016 y 2026

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

ign_sudam.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.4
)

ign_arg.plot(
    ax=ax,
    color="black",
    linewidth=0.5
)

gdf_arg.plot(
    ax=ax,
    column="mag",
    cmap="viridis",
    markersize=5,
    legend=True,
    legend_kwds={
        "orientation": "horizontal",
        "shrink": 0.4,
        "pad": 0.04
    }
)

# Zoom sobre Argentina y alrededores
ax.set_xlim(-76, -24)
ax.set_ylim(-90, -20)

ax.set_title(
    "Sismos ubicados dentro del área de estudio",
    fontsize=14,
    pad=12
)

plt.show()

**Figura 2. Sismos ubicados dentro del área de estudio.**

El mapa muestra los eventos cuyos epicentros se encuentran dentro del área de análisis. El color de los puntos representa la magnitud de cada sismo.

# 4. Exploración inicial del conjunto de datos


Se analiza la estructura general de los datos: cuántas filas y columnas hay, de qué tipo es cada variable, cuántos valores faltan y si hay registros duplicados.

### 4.1 Estructura y tipos de datos

In [ ]:
print("Filas:", df_arg.shape[0])
print("Columnas:", df_arg.shape[1])


In [ ]:
df_arg.head(3)

In [ ]:
df_arg.describe().round(3)

In [ ]:
df_arg.info()

### 4.2 Valores faltantes

In [ ]:
nulos = df_arg.isna().sum()
nulos[nulos>0]

In [ ]:
# Análisis porcentual de nulos

((nulos[nulos>0]/df_arg.shape[0])*100).round(3)

Nst presenta aproximadamente un 60 % de valores faltantes, por lo que se propone excluirla del análisis posterior. Para las restantes variables, los faltantes representan una proporción reducida y se propone imputarlos mediante la mediana durante la etapa de transformación.

### 4.3 Registros duplicados


In [ ]:
id = df_arg["id"].value_counts()
id[id>1]

In [ ]:
df_arg.duplicated().value_counts()

No existen filas ni identificadores duplicados.

### 4.4 Variables categóricas


#### 4.4.1 Última actualización del evento

In [ ]:
df_arg["updated"].value_counts()

#### 4.4.2 Fecha y hora exacta del evento

In [ ]:
df_arg["time"].value_counts()

Las columnas “updated” y “time” se transformarán a tipo DateTime en la etapa de transformación.

#### 4.4.3 Tipo de medición de magnitud

In [ ]:

df_arg["magType"].value_counts()

No presenta categorías o valores que requieran un tratamiento particular.

#### 4.4.4 Estado de revisión

In [ ]:
df_arg["status"].value_counts()

#### 4.4.5 Tipo de evento

In [ ]:
df_arg["type"].value_counts()

#### 4.4.6 Red sísmica de preferencia para el evento

In [ ]:
df_arg["net"].value_counts()

#### 4.4.7 Red sísmica que localizó el evento

In [ ]:
df_arg["locationSource"].value_counts()

#### 4.4.8 Red sísmica que calculó la magnitud

In [ ]:
df_arg["magSource"].value_counts()

Las variables categóricas con una categoría ampliamente predominante presentan poca variabilidad dentro del conjunto de datos y tienen una utilidad limitada para los análisis planteados. Por este motivo, se propone no considerarlas en el análisis posterior.

# 5. Análisis de magnitud y error de magnitud

La magnitud (`mag`) es una de las variables centrales del proyecto: mide la energía liberada por cada sismo y es la base para definir qué se considera actividad sísmica significativa en el modelo supervisado.

En esta sección se analiza:

- **La distribución de la magnitud**, para conocer el rango de valores y detectar registros atípicos.
- **El error de magnitud** (`magError`), para identificar mediciones con una incertidumbre desproporcionada que pudieran no ser confiables.

### 5.1 Distribución de la magnitud


In [ ]:
df_arg['mag'].value_counts().sort_index()

In [ ]:
ax = df_arg["mag"].hist(bins=50)

ax.set_title("Distribución de la magnitud de los sismos")
ax.set_xlabel("Magnitud")
ax.set_ylabel("Cantidad de eventos")

plt.show()

**Figura 3. Distribución de la magnitud de los sismos del área de análisis.**

El histograma muestra la cantidad de eventos registrados en cada intervalo de magnitud.

Análisis de variables atípicas

In [ ]:
index1 = df_arg[df_arg['mag']==5.06].index
index2 = df_arg[df_arg['mag']==5.18].index

In [ ]:
df_arg[df_arg["mag"].isin([5.06, 5.18])]

Se identificaron dos registros con valores de magnitud que resultan atípicos respecto del resto del conjunto de datos. Ambos presentan valores faltantes simultáneamente en nst, gap, dmin, rms, horizontalError y magNst, siendo estas variables de calidad. Al no poder evaluar la confiabilidad de las mediciones se recomienda excluir ambos registros del análisis posterior.

### 5.2 Error de magnitud


In [ ]:
bins = [0, 1, 2, 3, 4, 4.5, 5, 6, 7, 8, 9, 10]

mag_groups = pd.cut(
    bins=bins,
    df_arg['mag'],
    include_lowest=True
)


mag_error_summary = (
    df_arg
    .groupby(mag_groups, observed=True)['magError']
    .agg(
        count='count',
        median='median',
        q3=lambda x: x.quantile(0.75),
        max='max'
    )
)


mag_error_summary

In [ ]:
data_boxplot = [
    df_arg.loc[mag_groups == grupo, 'magError'].dropna()
    for grupo in mag_groups.cat.categories
]

labels = [
    str(grupo)
    for grupo in mag_groups.cat.categories
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    data_boxplot,
    tick_labels=labels
)

plt.xlabel('Magnitud')
plt.ylabel('Error de magnitud')
plt.title('Distribución de magError según magnitud')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Figura 4. Distribución del error de magnitud según el grupo de magnitud.**

Cada caja resume los valores de `magError` observados dentro de un intervalo de magnitud. La comparación permite evaluar si la incertidumbre de la medición varía entre eventos de distinta magnitud.

In [ ]:

df_arg[(df_arg["magError"]>=0.4) & (df_arg["mag"]>4) & (df_arg["mag"]<=4.5)]

In [ ]:
pd.set_option('display.max_columns', None)

df_arg[(df_arg["magError"]>=0.37) & (df_arg["mag"]>4.5) & (df_arg["mag"]<=5)]

No se identifican criterios suficientes para descartar los datos.

# 6. Análisis de profundidad y error de profundidad

La profundidad (`depth`) indica a cuántos kilómetros bajo la superficie se originó cada sismo. En Argentina es especialmente relevante: la placa de Nazca se hunde bajo la placa Sudamericana, así que hay sismos superficiales cerca de la cordillera y sismos muy profundos debajo del norte del país. Estos patrones sirven para diferenciar regiones en el modelo no supervisado.

En esta sección se analiza:

- **La distribución de la profundidad**, en general y en el espacio (gráfico 3D), para verificar que los valores tengan sentido físico.
- **El error de profundidad** (`depthError`), para identificar mediciones poco precisas.
- **Los registros con varios valores faltantes a la vez**, que aparecieron repetidamente durante el análisis y requieren una decisión.

### 6.1 Distribución de la profundidad


In [ ]:
ax = df_arg["depth"].hist(bins=50)

ax.set_title("Distribución de la profundidad de los sismos")
ax.set_xlabel("Profundidad (km)")
ax.set_ylabel("Cantidad de eventos")

plt.show()

**Figura 5. Distribución de la profundidad de los sismos del área de análisis.**

El histograma muestra la cantidad de eventos registrados en cada intervalo de profundidad, expresada en kilómetros.

In [ ]:

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    df_arg['longitude'],
    df_arg['latitude'],
    df_arg['depth'],
    c=df_arg['mag'],
    cmap='viridis',
    s=10,
    alpha=0.7
)

ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_zlabel('Profundidad (km)')

ax.invert_zaxis()

ax.view_init(elev=10, azim=-85)

ax.set_title('Distribución espacial de los eventos sísmicos')

fig.colorbar(
    scatter,
    ax=ax,
    label='Magnitud',
    shrink=0.7
)

plt.show()

Los eventos con profundidades superiores a 500 km se conservarán en el conjunto de datos, ya que su profundidad no constituye por sí misma un criterio suficiente para considerarlos registros anómalos.

### 6.2 Error de profundidad


In [ ]:
bins = [0, 10, 20, 40, 65, 100, 200, 300, 400, 500, 600, 650]

depth_groups = pd.cut(
    df_arg['depth'],
    bins=bins,
    include_lowest=True
)


depth_error_summary = (
    df_arg
    .groupby(depth_groups, observed=True)['depthError']
    .agg(
        count='count',
        median='median',
        q3=lambda x: x.quantile(0.75),
        max='max'
    )
)


depth_error_summary

In [ ]:
data_boxplot = [
    df_arg.loc[depth_groups == grupo, 'depthError'].dropna()
    for grupo in depth_groups.cat.categories
]

labels = [
    str(grupo)
    for grupo in depth_groups.cat.categories
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    data_boxplot,
    tick_labels=labels
)

plt.xlabel('Profundidad (km)')
plt.ylabel('Error de profundidad (km)')
plt.title('Distribución de depthError según profundidad')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

El gráfico compara el error estimado de la profundidad (depthError) entre distintos intervalos de profundidad de los sismos. En cada caja, la línea central representa la mediana y la altura de la caja muestra la dispersión del 50 % central de los registros. Las observaciones alejadas del resto pueden indicar eventos con un error de profundidad particularmente alto. Esta visualización permite evaluar cómo varía la precisión de la profundidad estimada según el intervalo analizado; no implica por sí misma una relación causal.

Observación de valores extremos

In [ ]:

df_arg[(df_arg["depthError"]==25) & (df_arg["depth"]<11)]

In [ ]:

df_arg[(df_arg["depthError"]>=15) & (df_arg["depth"]>20) & (df_arg["depth"]<41)]

In [ ]:

df_arg[(df_arg["depthError"]==25.700) & (df_arg["depth"]>100) & (df_arg["depth"]<201)]

In [ ]:
df_arg[(df_arg["depthError"]==32.700) & (df_arg["depth"]<301)]

No se identifican criterios suficientes para descartar los datos.

### 6.3 Análisis de registros con múltiples valores faltantes


Se analiza por la reiterada presencia de datos con varias columnas NaN.


In [ ]:

df_nan = df_arg.isna().sum(axis=1)

df_arg.loc[
    df_nan > 2, ["depth","depthError","mag","magError", "magNst","gap","dmin","rms","nst","horizontalError" ]
].assign(
    nan_count=df_nan
).sort_values(
    by='nan_count',
    ascending=False
)

In [ ]:
# cuales son los id de los registros con 5 o más valores NaN

nan_count = df_arg.isna().sum(axis=1)

registros_incompletos = df_arg[nan_count >= 5]

registros_incompletos["id"].tolist()

Los registros que presentan 5 o 6 variables NaN quedan practicamente sin variables de métrica de calidad, por lo que propongo descartarlos (notar que las primeras dos ya se había propuesto descartarlas). El resto tienen 3 métricas faltantes, como no todas tienen el mismo peso, veo que patrones existen:

In [ ]:
df_arg.loc[
    df_nan == 3,
    ['nst', 'gap', 'dmin', 'rms', 'horizontalError', 'depthError', 'magError', 'magNst']
].isna().astype(int).value_counts()


En un principio, según estos resultados, podrían mantenerse todas filas con 3 variables NaN.

# 7. Análisis de variables de calidad

Se revisan las variables que indican qué tan confiable es cada medición. Para cada una se muestra un resumen estadístico, un boxplot para ver los valores extremos y el detalle de los registros más altos, para decidir si son errores o datos válidos.


### 7.1 Error horizontal


In [ ]:
df_arg["horizontalError"].describe()

In [ ]:
plt.boxplot(df_arg["horizontalError"].dropna())
plt.ylabel("Error horizontal (km)")
plt.title("Distribución del error horizontal de la ubicación")
plt.show()

In [ ]:

df_arg[(df_arg["horizontalError"]>=25)]

### 7.2 Gap


In [ ]:
df_arg["gap"].describe()

In [ ]:
plt.boxplot(df_arg["gap"].dropna())
plt.ylabel("Brecha azimutal (°)")
plt.title("Distribución de la brecha azimutal de los sismos")
plt.show()

El gráfico muestra la distribución de la brecha azimutal (gap), una medida de la cobertura de estaciones sísmicas alrededor del epicentro. Valores elevados indican una distribución menos favorable de las estaciones y, por lo tanto, una posible menor confiabilidad en la localización. Los valores extremos se revisan posteriormente para identificar los registros que requieren atención.

In [ ]:
df_arg[(df_arg["gap"]>=290)]

### 7.3 Distancia mínima


In [ ]:
df_arg["dmin"].describe()

In [ ]:
plt.boxplot(df_arg["dmin"].dropna())
plt.ylabel("Distancia a la estación más cercana (°)")
plt.title("Distribución de la distancia mínima a una estación sísmica")
plt.show()

El gráfico muestra la distribución de dmin, que representa la distancia angular entre cada epicentro y la estación sísmica más cercana. Valores bajos indican una estación próxima al evento, mientras que valores altos reflejan una menor proximidad de estaciones y pueden asociarse con una localización menos precisa. La variable se interpreta como un indicador de calidad, no como una característica física del sismo.

### 7.4 RMS


In [ ]:
df_arg["rms"].describe()

In [ ]:
plt.boxplot(df_arg["rms"].dropna())
plt.ylabel("Error RMS (s)")
plt.title("Distribución del error RMS de la localización")
plt.show()

El gráfico representa la distribución del error RMS asociado a la localización de los sismos. Este indicador mide el ajuste entre los tiempos de llegada observados y los tiempos estimados por el modelo. Valores bajos sugieren una mayor concordancia, mientras que los valores altos pueden señalar registros cuya localización presenta mayor incertidumbre. Los valores extremos se revisan individualmente en el bloque siguiente.

In [ ]:
df_arg[(df_arg["rms"]>=2)]

### 7.5 Nst de magnitud

In [ ]:
df_arg["magNst"].describe()

In [ ]:
plt.boxplot(df_arg["magNst"].dropna())
plt.ylabel("Cantidad de estaciones")
plt.title("Distribución de estaciones utilizadas para calcular la magnitud")
plt.show()

El gráfico muestra la cantidad de estaciones sísmicas utilizadas para calcular la magnitud de los eventos. La distribución permite observar si la mayoría de las magnitudes se estimó con una cantidad similar de estaciones y detectar registros con valores excepcionalmente altos o bajos. Esta variable funciona como un indicador de calidad de la estimación de magnitud.

In [ ]:
df_arg[(df_arg["magNst"]>=600)]

Las variables de calidad, aunque presentan valores elevados, al observarlos en detalle no se encuentran otras particularidades que hagan dudar de las observaciones, por lo que no se recomienda descartar datos.

# 8. Análisis espacial



<p align="center">
  <img src="https://github.com/fer-gif/proyecto_ypf/blob/main/imagenes/placas_tectonicas_sudamerica.png?raw=1" width="800">
</p>

<p align="center">
  <em>Contexto tectónico de América del Sur y regiones circundantes.</em><br>
  <em>Fuente: Modificado de Bird, 2003.</em>
</p>

Por lo que se puede observar al comparar las imágenes de los apartados 3 y 6.1 con la imagen anterior, se pueden distinguir tres zonas sísmicas principales. La primera corresponde a la subducción de las placas de Nazca y Antártica por debajo de las placas Sudamericana y de Scotia. La segunda se caracteriza por el movimiento transformante asociado a la placa de Scotia y por la presencia predominante de sismos de poca profundidad. Por último, se identifica una tercera zona correspondiente a la subducción de la placa Sudamericana por debajo de la placa de Sandwich. En las zonas de subducción se observa además un rango de profundidades mayor.


# 9. Análisis temporal


### 9.1 Evolución anual



In [ ]:
horario = pd.to_datetime(df_arg['time']).dt.time
año = pd.to_datetime(df_arg['time']).dt.year
mes = pd.to_datetime(df_arg['time']).dt.month

In [ ]:

eventos_por_año = año.value_counts().sort_index()

print(eventos_por_año)

eventos_por_año.plot(kind="bar")

plt.xlabel("Año")
plt.ylabel("Cantidad de sismos")
plt.title("Cantidad anual de sismos registrados")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Se observa una distribución anual bastante homogénea salvo para el año 2021 que se estudiará por separado.

### 9.2 Distribución mensual


In [ ]:
eventos_por_mes = mes.value_counts().sort_index()

print(eventos_por_mes)

nombres_meses = [
    "Enero", "Febrero", "Marzo", "Abril",
    "Mayo", "Junio", "Julio", "Agosto",
    "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

eventos_por_mes.index = [nombres_meses[i - 1] for i in eventos_por_mes.index]

eventos_por_mes.plot(kind="bar")

plt.xlabel("Mes")
plt.ylabel("Cantidad de sismos")
plt.title("Cantidad mensual de sismos registrados")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Se observa una distribución mensual bastante homogenea salvo para el mes de agosto.
Se verá si el pico del año 2021 está relacionado con el pico de agosto.

In [ ]:
pd.crosstab(año, mes)

Efectivamente hay un pico considerable para agosto de 2021, veremos cuál es la distribución espacial del fenómeno.

In [ ]:

df_8_2021 = df_arg[(año == 2021) & (mes == 8)]

dia = pd.to_datetime(df_8_2021['time']).dt.day

gdf_8_2021 = gpd.GeoDataFrame(
    df_8_2021,
    geometry=gpd.points_from_xy(df_8_2021["longitude"], df_8_2021["latitude"]), crs="EPSG:4326"
)

In [ ]:

fig, ax = plt.subplots(figsize=(10, 12))

ign_sudam.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.4
)

ign_arg.plot(
    ax=ax,
    color="black",
    linewidth=0.5
)

gdf_8_2021.plot(
    ax=ax,
    column="mag",
    cmap="viridis",
    markersize=5,
    legend=True,
    legend_kwds={
        "orientation": "horizontal",
        "shrink": 0.4,
        "pad": 0.04
    }
)

# Zoom sobre Argentina y alrededores
ax.set_xlim(-76, -24)
ax.set_ylim(-90, -20)

ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title(
    "Distribución espacial de los sismos registrados en agosto de 2021",
    fontsize=14,
    pad=12
)

plt.show()

In [ ]:

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    df_8_2021['longitude'],
    df_8_2021['latitude'],
    df_8_2021['depth'],
    c=df_8_2021['mag'],
    cmap='viridis',
    s=10,
    alpha=0.7
)

ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_zlabel('Profundidad (km)')

ax.invert_zaxis()

ax.view_init(elev=10, azim=-85)

ax.set_title('Distribución de sismos 8/2021')

fig.colorbar(
    scatter,
    ax=ax,
    label='Magnitud',
    shrink=0.7
)

plt.show()

In [ ]:

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    df_8_2021['longitude'],
    df_8_2021['latitude'],
    df_8_2021['depth'],
    c=dia,
    cmap='viridis',
    s=10,
    alpha=0.7
)

ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_zlabel('Profundidad')

ax.invert_zaxis()

ax.view_init(elev=10, azim=-85)

ax.set_title('Distribución de sismos 8/2021')

fig.colorbar(
    scatter,
    ax=ax,
    label='Día',
    shrink=0.7
)

plt.show()

A partir de las imágenes se observan cientos de sismos en la zona de las Islas Sandwich durante el período de estudio, destacándose un evento de magnitud aproximada 8. Al investigar este episodio, se identificó que corresponde a un sismo principal seguido por réplicas registradas durante las semanas posteriores en el mismo sector.


# 10. Requerimientos para la transformación


En resumen se deberían hacer los siguientes cambios en la etapa de transformación:

* Descartar las columnas: "nst", "net", "type", "status", "locationSource", "magSource".
* Descartar las filas de id [iscgem621613005], [iscgem620210242], [us10007f0t], [us100073ln].
* Transformar las columnas "time" y "updated" a tipo DateTime.
* Rellenar los valores Na de las variables "gap", "dmin", "rms", "horizontalError", "magError", "magNst" con las medianas correspondientes.
* Crear columnas nuevas a partir de la variable "time".

# 11. Conclusiones de la EDA
La EDA permitió conocer las principales características del conjunto de datos e identificar algunos aspectos relevantes para el análisis. Los valores faltantes se concentran principalmente en las variables de calidad, y se identificaron algunos registros con varias de estas métricas ausentes. En cuanto a magnitud y profundidad, no se encontraron valores que por sí solos justificaran el descarte de registros. Por otro lado, el análisis temporal mostró una variación importante durante 2021, principalmente por el aumento de eventos registrado en agosto, concentrados en la zona de las Islas Sandwich. En conjunto, estos resultados permitieron identificar las principales características y particularidades del conjunto de datos.


Más información sobre las variables:

https://earthquake.usgs.gov/data/comcat/data-eventterms.php#nst

https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php